In [ ]:
from pathlib import Path
import json

# Resolve the project root when this notebook is run independently.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
results_dir = project_root / "results"
results_path = results_dir / "ranked_results.jsonl"
prediction_path = results_dir / "prediction.txt"
behaviors_path = project_root / "data/primary/test_set/behaviors.tsv"

with open(
    behaviors_path,
    "r",
    encoding="utf-8"
) as bf, open(
    results_path,
    "r",
    encoding="utf-8"
) as rf, open(
    prediction_path,
    "w",
    encoding="utf-8"
) as pf:

    for line_no, (behavior_line, result_line) in enumerate(
        zip(bf, rf), 1
    ):

        parts = behavior_line.rstrip("\n").split("\t")
        behavior_id = parts[0]

        impression_ids = [
            impression.split("-")[0]
            for impression in parts[4].split()
        ]

        result = json.loads(result_line)
        result_behavior_id = str(result["behavior_id"])

        if result_behavior_id != behavior_id:
            raise ValueError(
                f"Line {line_no}: Behavior ID mismatch: "
                f"{behavior_id} != {result_behavior_id}"
            )

        rank_lookup = {
            item["news_id"]: rank
            for rank, item in enumerate(result["ranking"], start=1)
        }

        try:
            ranks = [rank_lookup[news_id] for news_id in impression_ids]
        except KeyError as e:
            raise ValueError(
                f"Line {line_no}: Missing news_id {e} "
                f"in ranking of behavior {behavior_id}"
            )

        if sorted(ranks) != list(range(1, len(ranks) + 1)):
            raise ValueError(
                f"Line {line_no}: Invalid rank sequence "
                f"for behavior {behavior_id}: {ranks}"
            )

        pf.write(
            f"{behavior_id} "
            f"{json.dumps(ranks, separators=(',', ':'))}\n"
        )

        if line_no % 100_000 == 0:
            print(f"Converted: {line_no:,}")

print("Done.")
print("Output:", prediction_path)

In [ ]:
from pathlib import Path
import json

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
prediction_path = project_root / "results" / "prediction.txt"

total = 0
errors = 0

with open(prediction_path, "r", encoding="utf-8") as f:

    for line_no, line in enumerate(f, 1):

        try:
            behavior_id, ranks_json = line.rstrip("\n").split(maxsplit=1)
            ranks = json.loads(ranks_json)

            if not isinstance(ranks, list):
                raise ValueError("Ranks is not a list")

            if sorted(ranks) != list(range(1, len(ranks) + 1)):
                raise ValueError("Invalid rank sequence")

            total += 1

        except Exception as e:

            errors += 1

            if errors <= 10:
                print(f"ERROR line {line_no}: {e}")

print("========== PREDICTION VALIDATION ==========")
print("Rows:", total)
print("Errors:", errors)

In [ ]:
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
prediction_path = project_root / "results" / "prediction.txt"

with open(prediction_path, "r", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().rstrip())